# Cheatsheet — Klasifikasi Multi-Label

**Kapan pakai file ini:** satu dokumen boleh punya **beberapa label sekaligus**. Contoh: berita yang sekaligus ekonomi dan politik, tag artikel, gejala penyakit.

Alur tetap, tinggal ubah **§1 CONFIG**: struktur file, kolom, preprocessing, model.
Feature extraction dikunci di **TF-IDF** (default paling aman untuk klasifikasi teks).

```
intip file -> ambil kolom -> preprocessing -> TF-IDF -> model -> evaluasi -> output
```

Teori tiap langkah: `cheatsheet_klasifikasi_teks.ipynb` · Semua opsi: `latihan_sklearn.ipynb`

---
## §1 · CONFIG — ubah di sini saja

In [1]:
# ---------------- DATA: struktur file ----------------
PATH   = "data/berita_multilabel.csv"
SEP    = None
HEADER = "infer"
ENC    = None

# ---------------- DATA: kolom mana yang dipakai ----------------
TEXT_COL   = None       # None = deteksi otomatis
LABEL_COL  = "labels"   # kolom berisi label ganda, mis. "ekonomi|politik"
ID_COL     = None
LABEL_MAP  = None
MULTILABEL = True       # <- wajib True di file ini
PEMISAH    = "|"        # pemisah antar label di dalam satu sel

BAHASA = "id"

# ---------------- PREPROCESSING (True/False) ----------------
LOWERCASE   = True
MASK        = True
STOPWORD    = True
JAGA_NEGASI = True
STEMMING    = False

# ---------------- FEATURE EXTRACTION: TF-IDF ----------------
NGRAM  = (1, 2)
MIN_DF = 1

# ---------------- MODEL ----------------
MODEL       = "logreg"  # nb | logreg | svm | tree | rf
SEIMBANGKAN = True      # <- HAMPIR WAJIB untuk multilabel, lihat catatan di bawah

TEST_SIZE, SEED = 0.3, 42

---
## §2 · Intip struktur file dulu

Jalankan ini **sebelum** apa pun. Kalau nama kolom, separator, atau encoding-nya tidak seperti
dugaan, perbaiki di CONFIG lalu jalankan ulang sel ini. Jangan menebak struktur file.

In [2]:
import re, numpy as np, pandas as pd

# ---- 1. lihat 3 baris pertama file MENTAH (sebelum pandas menafsirkannya) ----
print("=== isi mentah 3 baris pertama ===")
with open(PATH, encoding="utf-8", errors="replace") as f:
    for i, baris in zip(range(3), f):
        print(f"  {i}: {baris.rstrip()[:110]}")

# ---- 2. baca dengan setelan di CONFIG ----
_sep = SEP or ("\t" if PATH.endswith((".tsv", ".tab")) else ",")
for _enc in ([ENC] if ENC else ["utf-8", "latin-1", "cp1252"]):
    try:
        raw = pd.read_csv(PATH, sep=_sep, header=HEADER, encoding=_enc,
                          engine="python", on_bad_lines="skip")
        break
    except (UnicodeDecodeError, UnicodeError):
        continue

print(f"\n=== terbaca: {raw.shape[0]} baris x {raw.shape[1]} kolom "
      f"(sep={_sep!r}, encoding={_enc}) ===")
print("nama kolom :", list(raw.columns))
print("\njumlah nilai unik per kolom (kolom label biasanya yang paling sedikit):")
print(raw.nunique().to_string())
print("\n=== 3 baris pertama ===")
print(raw.head(3).to_string())

# ---- 3. saran isian CONFIG -- periksa dulu, kalau benar tinggal disalin ----
_teks = [c for c in raw.columns if raw[c].map(lambda v: isinstance(v, str)).mean() > 0.5]
_tebak_teks = max(_teks, key=lambda c: raw[c].astype(str).str.len().mean()) if _teks else None
_kand = [(raw[c].nunique(), c) for c in raw.columns
         if c != _tebak_teks and 2 <= raw[c].nunique() <= 200]
_tebak_label = min(_kand)[1] if _kand else None

print("\n=== saran untuk CONFIG (salin kalau tebakannya benar) ===")
print(f"TEXT_COL   = {_tebak_teks!r}")
print(f"LABEL_COL  = {_tebak_label!r}")
if _tebak_label is not None:
    _nilai = sorted(map(str, raw[_tebak_label].dropna().unique()))
    print(f"# {len(_nilai)} nilai unik di kolom label: {_nilai[:8]}")

=== isi mentah 3 baris pertama ===
  0: text,labels
  1: Tim nasional menang tiga gol tanpa balas pada laga kualifikasi,olahraga
  2: Pelatih menyebut kondisi pemain membaik menjelang laga final,olahraga

=== terbaca: 88 baris x 2 kolom (sep=',', encoding=utf-8) ===
nama kolom : ['text', 'labels']

jumlah nilai unik per kolom (kolom label biasanya yang paling sedikit):
text      88
labels     9

=== 3 baris pertama ===
                                                                     text              labels
0          Tim nasional menang tiga gol tanpa balas pada laga kualifikasi            olahraga
1            Pelatih menyebut kondisi pemain membaik menjelang laga final            olahraga
2  Klub itu resmi mendatangkan pemain gelandang dengan kontrak tiga musim  olahraga|teknologi

=== saran untuk CONFIG (salin kalau tebakannya benar) ===
TEXT_COL   = 'text'
LABEL_COL  = 'labels'
# 9 nilai unik di kolom label: ['ekonomi', 'ekonomi|olahraga', 'ekonomi|politik', 'olahraga', 'olahr

---
## §3 · Ambil kolom teks & label

`TEXT_COL` / `LABEL_COL` diisi `None` = dideteksi otomatis (kolom teks = string dengan rata-rata
terpanjang, kolom label = kolom lain dengan nilai unik paling sedikit). Kalau tebakannya salah,
tulis sendiri di CONFIG — nama kolom (`"message"`) atau indeks angka (`5`) sama-sama bisa.

In [3]:
# ---- tentukan kolom; kalau CONFIG diisi None, dideteksi otomatis ----
text_col, label_col = TEXT_COL, LABEL_COL

if text_col is None:      # kolom teks = kolom berisi string dengan rata-rata terpanjang
    kand = [c for c in raw.columns if raw[c].map(lambda v: isinstance(v, str)).mean() > 0.5]
    if not kand:
        raise ValueError("kolom teks tidak terdeteksi -> isi TEXT_COL di CONFIG")
    text_col = max(kand, key=lambda c: raw[c].astype(str).str.len().mean())

if label_col is None:     # kolom label = kolom lain dengan nilai unik paling sedikit
    batas = 200 if MULTILABEL else 20
    kand = [(raw[c].nunique(), c) for c in raw.columns
            if c != text_col and 2 <= raw[c].nunique() <= batas]
    if not kand:
        raise ValueError("kolom label tidak terdeteksi -> isi LABEL_COL di CONFIG")
    label_col = min(kand)[1]

print(f"kolom teks  : {text_col!r}")
print(f"kolom label : {label_col!r}")
print(f"kolom id    : {ID_COL!r}" + ("  (tidak dipakai)" if ID_COL is None else ""))

# ---- rapikan jadi DataFrame dua/tiga kolom ----
petakan = {text_col: "text", label_col: "label"}
if ID_COL is not None:
    petakan[ID_COL] = "id"
df = raw.rename(columns=petakan)[list(petakan.values())].copy()

df["text"] = df["text"].astype(str).str.strip()
if not MULTILABEL:
    df["label"] = df["label"].apply(lambda v: v.strip().lower() if isinstance(v, str) else v)
if LABEL_MAP:
    df["label"] = df["label"].map(LABEL_MAP).fillna(df["label"])

n0 = len(df)
df = df[df["text"].str.len() >= 3]
df = df[~df["text"].str.lower().isin({"nan", "none", "na", "-"})]
df = df.dropna(subset=["text", "label"]).drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"\nsetelah dibersihkan: {n0} -> {len(df)} baris")

kolom teks  : 'text'
kolom label : 'labels'
kolom id    : None  (tidak dipakai)

setelah dibersihkan: 88 -> 88 baris


In [4]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

from sklearn.preprocessing import MultiLabelBinarizer

# "ekonomi|politik" -> ["ekonomi", "politik"] -> matriks 0/1, satu kolom per label
df["label_list"] = df["label"].map(lambda v: [x.strip() for x in str(v).split(PEMISAH) if x.strip()])
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df["label_list"])
KELAS = list(mlb.classes_)

print("label unik :", KELAS)
print("rata-rata label per dokumen:", round(Y.sum(axis=1).mean(), 2))
print()
print("contoh matriks target (5 baris pertama):")
print(pd.DataFrame(Y[:5], columns=KELAS).to_string(index=False))

label unik : ['ekonomi', 'olahraga', 'politik', 'teknologi']
rata-rata label per dokumen: 1.12

contoh matriks target (5 baris pertama):
 ekonomi  olahraga  politik  teknologi
       0         1        0          0
       0         1        0          0
       0         1        0          1
       0         1        0          0
       1         1        0          0


---
## §4 · Preprocessing

Semua sakelar di CONFIG dibaca di sini. Kalau semuanya `False`, teks diteruskan apa adanya —
`TfidfVectorizer` tetap melakukan lowercase dan tokenisasi sendiri, jadi pipeline tetap jalan.

In [5]:
STOP_EN = {"i","me","my","we","you","your","he","she","it","they","them","this","that","is","are",
           "was","were","be","been","have","has","had","do","does","did","a","an","the","and","but",
           "if","or","because","as","of","at","by","for","with","to","from","in","out","on","off",
           "then","so","than","too","very","just","now","s","t","can","will","there","here","what",
           "when","how","all","any","am","been","its","our","their"}
STOP_ID = {"yang","dan","di","ke","dari","ini","itu","untuk","dengan","pada","adalah","ada","saya",
           "kamu","dia","kami","kita","mereka","akan","sudah","telah","juga","atau","karena","agar",
           "saja","oleh","sebagai","dalam","para","nya","banget","sekali","sangat","tetapi","tapi"}
NEGASI  = {"no","not","never","nor","cannot"} | {"tidak","bukan","tanpa","jangan","belum","kurang"}

stop = (STOP_EN if BAHASA == "en" else STOP_ID)
if JAGA_NEGASI:
    stop = stop - NEGASI

if STEMMING and BAHASA == "en":
    from nltk.stem import PorterStemmer
    _stem = PorterStemmer().stem
elif STEMMING and BAHASA == "id":
    try:
        from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
        _stem = StemmerFactory().create_stemmer().stem
    except ImportError:
        print("Sastrawi belum terpasang -> stemming dilewati")
        _stem = None
else:
    _stem = None


def bersihkan(teks):
    t = str(teks)
    if LOWERCASE:
        t = t.lower()
    if MASK:                                        # URL, mention, angka -> token generik
        t = re.sub(r"http\S+|www\.\S+|\b\S+\.(?:com|org|net|ly|id|co)\S*", " urltoken ", t)
        t = re.sub(r"@\w+", " usertoken ", t)
        t = re.sub(r"\b\d+\b", " numtoken ", t)
    kata = re.findall(r"[a-zA-Z]+", t)
    if STOPWORD:
        kata = [w for w in kata if w not in stop]
    if _stem:
        kata = [_stem(w) for w in kata]
    return " ".join(kata) if kata else "kosongtoken"


df["clean"] = df["text"].map(bersihkan)
print("sebelum:", df["text"].iloc[0][:70])
print("sesudah:", df["clean"].iloc[0][:70])

sebelum: Tim nasional menang tiga gol tanpa balas pada laga kualifikasi
sesudah: tim nasional menang tiga gol tanpa balas laga kualifikasi


---
## §5 · Split train/test

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], Y, test_size=TEST_SIZE, random_state=SEED)
print("train:", X_train.shape[0], "| test:", X_test.shape[0])

train: 61 | test: 27


---
## §6 · Model

In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

bobot = "balanced" if SEIMBANGKAN else None
PILIHAN_MODEL = {
    "nb":     MultinomialNB(),
    "logreg": LogisticRegression(max_iter=1000, class_weight=bobot),
    "svm":    LinearSVC(class_weight=bobot),
    "tree":   DecisionTreeClassifier(max_depth=8, class_weight=bobot, random_state=SEED),
    "rf":     RandomForestClassifier(n_estimators=200, class_weight=bobot, random_state=SEED),
}
clf = PILIHAN_MODEL[MODEL]
print("model:", clf.__class__.__name__)

model: LogisticRegression


---
## §7 · TF-IDF + latih

`Pipeline` menjamin vectorizer hanya di-`fit` pada data latih — kalau tidak, nilai IDF ikut
"melihat" data uji dan skornya jadi palsu.

In [8]:
from sklearn.multiclass import OneVsRestClassifier

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=NGRAM, min_df=MIN_DF, sublinear_tf=True)),
    ("clf",   OneVsRestClassifier(clf)),      # satu classifier per label
])
model.fit(X_train, y_train)
print("selesai dilatih")

selesai dilatih


---
## §8 · Evaluasi

In [9]:
pred = model.predict(X_test)

print("f1 micro:", round(f1_score(y_test, pred, average="micro", zero_division=0), 3),
      "  <- dihitung dari total TP/FP/FN semua label")
print("f1 macro:", round(f1_score(y_test, pred, average="macro", zero_division=0), 3),
      "  <- rata-rata f1 antar label\n")
print(classification_report(y_test, pred, target_names=KELAS, zero_division=0))

# subset accuracy: berapa dokumen yang SEMUA labelnya tepat
tepat = (pred == y_test).all(axis=1).mean()
print("subset accuracy (semua label tepat):", round(tepat, 3))

# label yang paling sering terlewat
lewat = ((y_test == 1) & (pred == 0)).sum(axis=0)
print("\nlabel paling sering terlewat:",
      dict(zip(KELAS, lewat)))

f1 micro: 0.667   <- dihitung dari total TP/FP/FN semua label
f1 macro: 0.662   <- rata-rata f1 antar label

              precision    recall  f1-score   support

     ekonomi       0.67      0.67      0.67         6
    olahraga       1.00      0.50      0.67         8
     politik       1.00      0.56      0.71         9
   teknologi       0.75      0.50      0.60         6

   micro avg       0.84      0.55      0.67        29
   macro avg       0.85      0.56      0.66        29
weighted avg       0.88      0.55      0.67        29
 samples avg       0.59      0.57      0.58        29

subset accuracy (semua label tepat): 0.556

label paling sering terlewat: {'ekonomi': np.int64(2), 'olahraga': np.int64(4), 'politik': np.int64(4), 'teknologi': np.int64(3)}


---
## §9 · Prediksi teks baru

In [10]:
teks_baru = ["pemerintah menaikkan anggaran subsidi energi tahun depan",
             "klub sepak bola itu meraih pendanaan investasi dari luar negeri"]

hasil = model.predict([bersihkan(t) for t in teks_baru])
for t, baris in zip(teks_baru, hasil):
    label = [KELAS[j] for j in np.where(baris)[0]]
    print(f"  {label} <- {t[:58]}")

  ['ekonomi', 'politik'] <- pemerintah menaikkan anggaran subsidi energi tahun depan
  [] <- klub sepak bola itu meraih pendanaan investasi dari luar n


---
## §10 · Output — simpan model & tulis hasil ke file

Tiga keluaran yang biasanya diminta: **model tersimpan**, **file hasil prediksi**, dan
**ringkasan skor**. Model disimpan bersama setelan preprocessing-nya — tanpa itu, model yang
dimuat ulang tidak tahu teks baru harus dibersihkan bagaimana.

In [11]:
import joblib, json

NAMA = PATH.split("/")[-1].split(".")[0]

joblib.dump({"pipeline": model, "mlb": mlb,
             "prep": dict(BAHASA=BAHASA, LOWERCASE=LOWERCASE, MASK=MASK,
                          STOPWORD=STOPWORD, JAGA_NEGASI=JAGA_NEGASI, STEMMING=STEMMING)},
            f"model_{NAMA}.joblib")

# multilabel: kembalikan matriks 0/1 jadi teks "a|b" supaya file hasilnya terbaca manusia
ke_teks = lambda baris: PEMISAH.join([KELAS[j] for j in np.where(baris)[0]])
hasil = pd.DataFrame({"text": X_test.values,
                      "aktual":   [ke_teks(r) for r in y_test],
                      "prediksi": [ke_teks(r) for r in pred]})
hasil["tepat_semua"] = hasil["aktual"] == hasil["prediksi"]
hasil.to_csv(f"hasil_prediksi_{NAMA}.csv", index=False)

ringkas = {
    "dataset": PATH, "model": MODEL, "case": "multilabel",
    "fitur": f"TF-IDF ngram={NGRAM} min_df={MIN_DF}", "label": KELAS,
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "f1_micro": round(float(f1_score(y_test, pred, average="micro", zero_division=0)), 4),
    "f1_macro": round(float(f1_score(y_test, pred, average="macro", zero_division=0)), 4),
    "subset_accuracy": round(float((pred == y_test).all(axis=1).mean()), 4),
}
with open(f"ringkasan_{NAMA}.json", "w") as f:
    json.dump(ringkas, f, indent=2)

print("tersimpan:")
print(f"  model_{NAMA}.joblib")
print(f"  hasil_prediksi_{NAMA}.csv   <- {len(hasil)} baris")
print(f"  ringkasan_{NAMA}.json")
print()
print(json.dumps(ringkas, indent=2))
print()
print(hasil.head(4).to_string(index=False))

tersimpan:
  model_berita_multilabel.joblib
  hasil_prediksi_berita_multilabel.csv   <- 27 baris
  ringkasan_berita_multilabel.json

{
  "dataset": "data/berita_multilabel.csv",
  "model": "logreg",
  "case": "multilabel",
  "fitur": "TF-IDF ngram=(1, 2) min_df=1",
  "label": [
    "ekonomi",
    "olahraga",
    "politik",
    "teknologi"
  ],
  "n_train": 61,
  "n_test": 27,
  "f1_micro": 0.6667,
  "f1_macro": 0.6619,
  "subset_accuracy": 0.5556
}

                                                                     text    aktual  prediksi  tepat_semua
 presiden mengumumkan perombakan kabinet setelah evaluasi kinerja menteri   politik                  False
                tim nasional menang tiga gol tanpa balas laga kualifikasi  olahraga  olahraga         True
  pembaruan perangkat lunak menambal celah keamanan dilaporkan bulan lalu teknologi teknologi         True
perusahaan teknologi merilis ponsel pintar terbaru kamera resolusi tinggi teknologi teknologi         True


---
## Catatan khusus case multi-label

Ini satu-satunya case yang **strukturnya beda**, bukan cuma metriknya:

| Hal | Single-label | Multi-label |
|---|---|---|
| Target `y` | satu kolom berisi nama kelas | matriks 0/1, satu kolom per label |
| Encoder | tidak perlu | `MultiLabelBinarizer` |
| Classifier | langsung | dibungkus `OneVsRestClassifier` (satu model per label) |
| `stratify` saat split | bisa | **tidak bisa** (tidak ada kelas tunggal untuk distratifikasi) |
| Metrik utama | `f1_macro` | `f1_micro` + `f1_macro` |
| Prediksi | satu nama kelas | daftar label (bisa kosong, bisa banyak) |

- **`f1_micro` vs `f1_macro`**: micro menjumlahkan TP/FP/FN semua label lalu menghitung sekali —
  label yang sering muncul mendominasi. Macro merata-ratakan f1 tiap label — label langka
  ikut menentukan. Laporkan keduanya.
- **Subset accuracy** (semua label tepat) selalu jauh lebih rendah dari f1, dan itu wajar:
  salah satu label saja sudah membuat dokumen dihitung salah total.
### Kenapa `SEIMBANGKAN = True` hampir wajib di sini

`OneVsRestClassifier` memecah masalah jadi satu classifier biner per label: "ekonomi vs bukan
ekonomi", "politik vs bukan politik", dan seterusnya. Tiap sub-masalah itu **otomatis timpang** —
kalau ada 4 label, satu label rata-rata cuma muncul di seperempat dokumen. Akibatnya model
belajar bahwa menjawab "bukan" hampir selalu aman, dan prediksinya jadi kosong semua.

Angka nyata dari dataset di notebook ini:

| Model | f1 micro | prediksi kosong |
|---|---|---|
| `svm` | 0.49 | 56% |
| `logreg` tanpa `balanced` | **0.00** | **100%** |
| `logreg` + `balanced` | **0.67** | 30% |

`logreg` polos memprediksi **tidak ada label sama sekali** untuk seluruh dokumen uji — dan
akurasinya tetap terlihat "lumayan" kalau diukur per sel matriks. Itu jebakan yang sama dengan
case imbalanced, hanya muncul dalam bentuk berbeda.

- Prediksi masih bisa **kosong** untuk sebagian dokumen. Kalau itu mengganggu, pakai
  `predict_proba` lalu ambil label dengan skor tertinggi sebagai jaring pengaman:

```python
prob = model.predict_proba(X_test)
pred_min1 = pred.copy()
kosong = pred.sum(axis=1) == 0
pred_min1[kosong, prob[kosong].argmax(axis=1)] = 1     # paksa minimal satu label
```